In [1]:
from pathlib import Path
import pandas as pd

In [6]:
data_path = Path("../data/smartdoc_demo")
new_documents_path = Path("../data/new_documents")

In [7]:
categories = [
    folder.name
    for folder in data_path.iterdir()
    if folder.is_dir()
]

print(categories)

['activites_voyage', 'agriculture_urbanisme', 'algorithmes_structures', 'assurance', 'astronomie', 'banque_credit', 'bases_de_donnees', 'biodiversite_ecosystemes', 'bonnes_pratiques_dev', 'budget_epargne', 'climat', 'destinations', 'developpement_web', 'energie', 'fiscalite_retraite', 'gastronomie_voyage', 'hebergement_voyage', 'intelligence_artificielle', 'investissement', 'planification_voyage', 'transport_voyage', 'voyages']


In [8]:
# Reconstruire les données depuis les dossiers réels
documents = []

for category_path in data_path.iterdir():

    if not category_path.is_dir():
        continue

    category = category_path.name

    for file_path in category_path.glob("*.txt"):

        text = file_path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        documents.append({
            "filename": file_path.name,
            "text": text,
            "category": category,
            "path": str(file_path)
        })

df = pd.DataFrame(documents)

print("Nombre total de documents :", len(df))
print("\nDocuments par catégorie :")
print(df["category"].value_counts())

Nombre total de documents : 41

Documents par catégorie :
category
biodiversite_ecosystemes     4
agriculture_urbanisme        3
algorithmes_structures       3
bonnes_pratiques_dev         3
investissement               3
banque_credit                2
budget_epargne               2
climat                       2
destinations                 2
developpement_web            2
fiscalite_retraite           2
planification_voyage         2
voyages                      2
activites_voyage             1
assurance                    1
astronomie                   1
bases_de_donnees             1
energie                      1
gastronomie_voyage           1
hebergement_voyage           1
intelligence_artificielle    1
transport_voyage             1
Name: count, dtype: int64


In [9]:
print(df["text"].str.len())

0     1284
1     1426
2     1162
3     1443
4     1357
5     1089
6     1395
7     1353
8     1438
9     1061
10    1354
11    1090
12    1467
13    1387
14    1377
15    1331
16    1367
17    1373
18    1368
19    1033
20    1352
21    1139
22    1434
23    1388
24    1030
25    1114
26    1400
27    1154
28    1322
29    1324
30    1363
31    1055
32    1378
33    1409
34    1109
35    1355
36    1339
37    1400
38    1269
39    1227
40    1015
Name: text, dtype: int64


In [10]:
print(df.iloc[0]["text"][:300])

Une randonnée réussie se mesure surtout au plaisir partagé et au retour en sécurité.
Une randonnée en montagne se prépare en fonction de l'itinéraire, de la météo et du niveau réel de chaque participant. Une carte précise et une trace hors ligne permettent de s'orienter même sans réseau. Il faut vér


In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
embeddings = embedding_model.encode(
    df["text"].tolist(),
    show_progress_bar=True
)

print("Shape :", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Shape : (41, 768)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarities = cosine_similarity(
    embeddings[0].reshape(1, -1),
    embeddings
)[0]

top_indices = np.argsort(similarities)[::-1]

for index in top_indices[:5]:
    print(
        f"{similarities[index]:.3f} | "
        f"{df.iloc[index]['category']} | "
        f"{df.iloc[index]['filename']}"
    )

1.000 | activites_voyage | randonnee_montagne.txt
0.532 | planification_voyage | budget_voyage.txt
0.522 | planification_voyage | tourisme_responsable.txt
0.458 | voyages | voyage_avec_enfants.txt
0.439 | biodiversite_ecosystemes | forets.txt


In [14]:
new_document_path = Path("../data/new_documents/apprentissage_python.txt")

new_text = new_document_path.read_text(
    encoding="utf-8",
    errors="ignore"
)

new_embedding = embedding_model.encode(
    [new_text]
)

print("Embedding shape :", new_embedding.shape)

Embedding shape : (1, 768)


In [15]:
similarities = cosine_similarity(
    new_embedding,
    embeddings
)[0]

results = []

for i, score in enumerate(similarities):
    results.append({
        "filename": df.iloc[i]["filename"],
        "category": df.iloc[i]["category"],
        "similarity": score
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "similarity",
    ascending=False
)

print(results_df.head(10).to_string(index=False))

                     filename                  category  similarity
                   python.txt    algorithmes_structures    0.710734
intelligence_artificielle.txt intelligence_artificielle    0.411179
           tests_logiciel.txt      bonnes_pratiques_dev    0.353601
       structures_donnees.txt    algorithmes_structures    0.318272
            securite_code.txt      bonnes_pratiques_dev    0.314011
       gastronomie_voyage.txt        gastronomie_voyage    0.296909
        developpement_web.txt         developpement_web    0.291966
            budget_voyage.txt      planification_voyage    0.286213
             biodiversite.txt  biodiversite_ecosystemes    0.272372
          algorithmes_tri.txt    algorithmes_structures    0.259023


In [16]:
category_scores = (
    results_df
    .groupby("category")["similarity"]
    .mean()
    .sort_values(ascending=False)
)

print(category_scores)

category
algorithmes_structures       0.429343
intelligence_artificielle    0.411179
bonnes_pratiques_dev         0.299580
gastronomie_voyage           0.296909
planification_voyage         0.256009
voyages                      0.230455
bases_de_donnees             0.223988
developpement_web            0.223375
budget_epargne               0.201149
banque_credit                0.180341
biodiversite_ecosystemes     0.169991
transport_voyage             0.169100
activites_voyage             0.160815
agriculture_urbanisme        0.157476
fiscalite_retraite           0.153039
energie                      0.151066
hebergement_voyage           0.144448
destinations                 0.142405
investissement               0.133914
astronomie                   0.126157
climat                       0.094092
assurance                    0.092785
Name: similarity, dtype: float32


In [17]:
print(df[["filename", "category"]])
print("\nLongueur des textes :")
print(df["text"].str.len())

                         filename                   category
0          randonnee_montagne.txt           activites_voyage
1         agriculture_durable.txt      agriculture_urbanisme
2                   recyclage.txt      agriculture_urbanisme
3           urbanisme_durable.txt      agriculture_urbanisme
4             algorithmes_tri.txt     algorithmes_structures
5                      python.txt     algorithmes_structures
6          structures_donnees.txt     algorithmes_structures
7        assurance_habitation.txt                  assurance
8                 exoplanetes.txt                 astronomie
9                      banque.txt              banque_credit
10          credit_immobilier.txt              banque_credit
11                  databases.txt           bases_de_donnees
12               biodiversite.txt   biodiversite_ecosystemes
13                  eau_douce.txt   biodiversite_ecosystemes
14                     forets.txt   biodiversite_ecosystemes
15            ocean_plas

In [19]:
category_scores = (
    results_df
    .groupby("category")["similarity"]
    .mean()
    .sort_values(ascending=False)
)

print(category_scores)

category
algorithmes_structures       0.429343
intelligence_artificielle    0.411179
bonnes_pratiques_dev         0.299580
gastronomie_voyage           0.296909
planification_voyage         0.256009
voyages                      0.230455
bases_de_donnees             0.223988
developpement_web            0.223375
budget_epargne               0.201149
banque_credit                0.180341
biodiversite_ecosystemes     0.169991
transport_voyage             0.169100
activites_voyage             0.160815
agriculture_urbanisme        0.157476
fiscalite_retraite           0.153039
energie                      0.151066
hebergement_voyage           0.144448
destinations                 0.142405
investissement               0.133914
astronomie                   0.126157
climat                       0.094092
assurance                    0.092785
Name: similarity, dtype: float32


In [21]:
document_embeddings = embedding_model.encode(
    df["text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [22]:
new_document_path = Path("../data/new_documents/astronomie.txt")

astronomie_text = new_document_path.read_text(
    encoding="utf-8",
    errors="ignore"
)

astronomie_embedding = embedding_model.encode(
    [astronomie_text],
    normalize_embeddings=True
)

astronomie_similarities = cosine_similarity(
    astronomie_embedding,
    document_embeddings
)[0]

astronomie_results = []

for i, score in enumerate(astronomie_similarities):
    astronomie_results.append({
        "filename": df.iloc[i]["filename"],
        "category": df.iloc[i]["category"],
        "similarity": score
    })

astronomie_results_df = pd.DataFrame(
    astronomie_results
).sort_values(
    "similarity",
    ascending=False
)

print(
    astronomie_results_df.head(10)
    .to_string(index=False)
)

                 filename                 category  similarity
          exoplanetes.txt               astronomie    0.618009
         biodiversite.txt biodiversite_ecosystemes    0.364490
      energie_solaire.txt                  energie    0.330175
changement_climatique.txt                   climat    0.327836
       culture_locale.txt             destinations    0.308254
                hotel.txt       hebergement_voyage    0.298566
 tourisme_responsable.txt     planification_voyage    0.294948
    urbanisme_durable.txt    agriculture_urbanisme    0.291741
   gastronomie_voyage.txt       gastronomie_voyage    0.278672
        pollution_air.txt                   climat    0.277325


In [23]:
astronomie_similarities = cosine_similarity(
    astronomie_embedding,
    document_embeddings
)[0]

astronomie_results = []

for i, score in enumerate(astronomie_similarities):
    astronomie_results.append({
        "filename": df.iloc[i]["filename"],
        "category": df.iloc[i]["category"],
        "similarity": score
    })

astronomie_results_df = pd.DataFrame(
    astronomie_results
).sort_values(
    "similarity",
    ascending=False
)

print(
    astronomie_results_df.head(10)
    .to_string(index=False)
)

                 filename                 category  similarity
          exoplanetes.txt               astronomie    0.618009
         biodiversite.txt biodiversite_ecosystemes    0.364490
      energie_solaire.txt                  energie    0.330175
changement_climatique.txt                   climat    0.327836
       culture_locale.txt             destinations    0.308254
                hotel.txt       hebergement_voyage    0.298566
 tourisme_responsable.txt     planification_voyage    0.294948
    urbanisme_durable.txt    agriculture_urbanisme    0.291741
   gastronomie_voyage.txt       gastronomie_voyage    0.278672
        pollution_air.txt                   climat    0.277325


In [24]:
category_embeddings  = {}

for category in df["category"].unique():

    mask = df["category"].values == category

    category_embedding = document_embeddings[mask]

    category_embeddings[category] = category_embedding

print("Catégories :", list(category_embeddings.keys()))

Catégories : ['activites_voyage', 'agriculture_urbanisme', 'algorithmes_structures', 'assurance', 'astronomie', 'banque_credit', 'bases_de_donnees', 'biodiversite_ecosystemes', 'bonnes_pratiques_dev', 'budget_epargne', 'climat', 'destinations', 'developpement_web', 'energie', 'fiscalite_retraite', 'gastronomie_voyage', 'hebergement_voyage', 'intelligence_artificielle', 'investissement', 'planification_voyage', 'transport_voyage', 'voyages']


In [25]:
def create_category(category_name, document_embedding):

    category_embeddings[category_name] = [
        document_embedding
    ]

    category_path = data_path / category_name
    category_path.mkdir(
        parents=True,
        exist_ok=True
    )

    print(f"🆕 Nouvelle catégorie créée : {category_name}")
    print(f"📁 Dossier : {category_path}")

In [27]:
def find_best_category(document_embedding):

    best_category = None
    best_score = -1
    best_document = None

    for category, category_embeddings_list in category_embeddings.items():

        scores = cosine_similarity(
            document_embedding.reshape(1, -1),
            category_embeddings_list
        )[0]

        max_index = np.argmax(scores)
        max_score = float(scores[max_index])

        if max_score > best_score:

            best_score = max_score
            best_category = category

            # Pour les catégories existantes
            if category in df["category"].values:

                category_df = df[
                    df["category"] == category
                ].reset_index(drop=True)

                best_document = category_df.iloc[max_index]["filename"]

            else:

                best_document = "document de référence"

    return best_category, best_score, best_document

In [28]:
python_embedding = embedding_model.encode(
    [new_text],
    normalize_embeddings=True
)[0]

print(category_embeddings.keys())
print(len(category_embeddings))
category, score, document = find_best_category(
    python_embedding
)

print("Catégorie :", category)
print("Document le plus proche :", document)
print("Score :", round(score, 3))

dict_keys(['activites_voyage', 'agriculture_urbanisme', 'algorithmes_structures', 'assurance', 'astronomie', 'banque_credit', 'bases_de_donnees', 'biodiversite_ecosystemes', 'bonnes_pratiques_dev', 'budget_epargne', 'climat', 'destinations', 'developpement_web', 'energie', 'fiscalite_retraite', 'gastronomie_voyage', 'hebergement_voyage', 'intelligence_artificielle', 'investissement', 'planification_voyage', 'transport_voyage', 'voyages'])
22
Catégorie : algorithmes_structures
Document le plus proche : python.txt
Score : 0.711


In [29]:
category, score, document = find_best_category(
    astronomie_embedding[0]
)

print("Catégorie :", category)
print("Document le plus proche :", document)
print("Score :", round(score, 3))

Catégorie : astronomie
Document le plus proche : exoplanetes.txt
Score : 0.618


In [30]:
import ollama

def generate_category_name(text, max_chars=800):
    """
    Utilise Qwen2.5:3b en local (via Ollama) pour générer 
    un nom de catégorie court à partir du contenu d'un document.
    """

    excerpt = text[:max_chars]

    prompt = f"""Voici un extrait d'un document :

---
{excerpt}
---

Donne un nom de catégorie court (2 à 3 mots maximum) qui résume 
le sujet principal de ce document, en français.

Règles strictes :
- Réponds UNIQUEMENT avec le nom de la catégorie, rien d'autre
- Pas de phrase, pas d'explication, pas de ponctuation, pas de guillemets
- Utilise une forme générale (ex: "Recettes" et non "Recette de pâtes")

Nom de catégorie :"""

    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.2}  # basse température = plus stable/déterministe
    )

    category_name = response["message"]["content"].strip()

    # Nettoyage : les modèles locaux ont tendance à ajouter du texte parasite
    category_name = category_name.split("\n")[0]           # garde la 1ère ligne seulement
    category_name = category_name.strip('"\'.,: ')          # enlève guillemets/ponctuation
    category_name = category_name.replace("/", "-").replace("\\", "-")

    if len(category_name.split()) > 4:
        # Fallback : prend les 2 premiers mots significatifs
        category_name = " ".join(category_name.split()[:2])

    return category_name.capitalize()

In [31]:
from difflib import SequenceMatcher

def find_similar_existing_category_name(new_name, existing_categories, threshold=0.85):
    """
    Vérifie si un nom très proche existe déjà (évite les doublons
    du type "Voyage" vs "Voyages").
    """

    for existing in existing_categories:
        similarity = SequenceMatcher(
            None, new_name.lower(), existing.lower()
        ).ratio()
        if similarity >= threshold:
            return existing

    return None

In [32]:
def smartdoc_categorize(
    file_path,
    threshold=0.55
):

    file_path = Path(file_path)


    # --------------------------------------------------------
    # Vérifier que le fichier existe
    # --------------------------------------------------------

    if not file_path.exists():

        print(
            "❌ Fichier introuvable :",
            file_path
        )

        return None


    # --------------------------------------------------------
    # Lire le contenu
    # --------------------------------------------------------

    text = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )


    # --------------------------------------------------------
    # Vérifier que le document n'est pas vide
    # --------------------------------------------------------

    if not text.strip():

        print(
            "❌ Le document est vide :",
            file_path.name
        )

        return None


    # --------------------------------------------------------
    # Créer l'embedding
    # --------------------------------------------------------

    embedding = embedding_model.encode(
        [text],
        normalize_embeddings=True
    )[0]


    # --------------------------------------------------------
    # Trouver la meilleure catégorie
    # --------------------------------------------------------

    (
        category,
        score,
        closest_document
    ) = find_best_category(
        embedding
    )


    print("\n" + "=" * 55)

    print(
        "Document :",
        file_path.name
    )

    print(
        "Document le plus proche :",
        closest_document
    )

    print(
        "Meilleure catégorie :",
        category
    )

    print(
        "Score :",
        round(score, 3)
    )


    # ========================================================
    # CAS 1 : CATEGORIE EXISTANTE
    # ========================================================

    if score >= threshold:

        destination_category = category


        # Ajouter le nouvel embedding
        # à la catégorie existante

        category_embeddings[
            category
        ].append(
            embedding
        )


        print(
            "→ Catégorie existante :",
            category
        )


        action = "existing_category"


    # ========================================================
    # CAS 2 : NOUVELLE CATEGORIE (via LLM)
    # ========================================================

    else:

        # Demander au LLM local un nom de catégorie
        # pertinent basé sur le contenu du document

        proposed_name = generate_category_name(text)

        print(
            "🤖 Nom proposé par le LLM :",
            proposed_name
        )


        # Vérifier qu'un nom très proche n'existe pas déjà
        # (évite "Voyage" vs "Voyages")

        existing_match = find_similar_existing_category_name(
            proposed_name,
            category_embeddings.keys()
        )


        if existing_match:

            destination_category = existing_match

            print(
                "→ Nom proche d'une catégorie existante, "
                "fusion avec :",
                existing_match
            )

            category_embeddings[
                existing_match
            ].append(
                embedding
            )

            action = "existing_category"


        else:

            destination_category = proposed_name

            create_category(
                destination_category,
                embedding
            )

            print(
                "→ Nouvelle catégorie créée :",
                destination_category
            )

            action = "new_category"


    # ========================================================
    # DEPLACER LE DOCUMENT
    # ========================================================

    destination_folder = (
        data_path / destination_category
    )


    destination_folder.mkdir(
        parents=True,
        exist_ok=True
    )


    destination_file = (
        destination_folder / file_path.name
    )


    file_path.rename(
        destination_file
    )


    print(
        "📁 Fichier déplacé vers :",
        destination_file
    )


    # ========================================================
    # RESULTAT
    # ========================================================

    return {

        "category": destination_category,

        "score": float(score),

        "action": action,

        "path": str(destination_file)
    }

In [33]:
result = smartdoc_categorize(
    "../data/new_documents/astronomie.txt"
)

print(result)


Document : astronomie.txt
Document le plus proche : exoplanetes.txt
Meilleure catégorie : astronomie
Score : 0.618


AttributeError: 'numpy.ndarray' object has no attribute 'append'

In [34]:
result_python = smartdoc_categorize(
    "../data/new_documents/apprentissage_python.txt"
)

print(result_python)


Document : apprentissage_python.txt
Document le plus proche : python.txt
Meilleure catégorie : algorithmes_structures
Score : 0.711


AttributeError: 'numpy.ndarray' object has no attribute 'append'

In [35]:
print(category_embeddings.keys())

dict_keys(['activites_voyage', 'agriculture_urbanisme', 'algorithmes_structures', 'assurance', 'astronomie', 'banque_credit', 'bases_de_donnees', 'biodiversite_ecosystemes', 'bonnes_pratiques_dev', 'budget_epargne', 'climat', 'destinations', 'developpement_web', 'energie', 'fiscalite_retraite', 'gastronomie_voyage', 'hebergement_voyage', 'intelligence_artificielle', 'investissement', 'planification_voyage', 'transport_voyage', 'voyages'])


In [37]:
category_embeddings = {}

for category in df["category"].unique():

    mask = df["category"].values == category

    category_embeddings[category] = (
        document_embeddings[mask]
    )

print(category_embeddings.keys())

dict_keys(['activites_voyage', 'agriculture_urbanisme', 'algorithmes_structures', 'assurance', 'astronomie', 'banque_credit', 'bases_de_donnees', 'biodiversite_ecosystemes', 'bonnes_pratiques_dev', 'budget_epargne', 'climat', 'destinations', 'developpement_web', 'energie', 'fiscalite_retraite', 'gastronomie_voyage', 'hebergement_voyage', 'intelligence_artificielle', 'investissement', 'planification_voyage', 'transport_voyage', 'voyages'])


In [38]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Matrice de similarité entre TOUS les documents
similarity_matrix = cosine_similarity(document_embeddings)

same_category_scores = []
different_category_scores = []

n = len(df)

for i in range(n):
    for j in range(i + 1, n):  # évite les doublons et l'auto-comparaison

        score = similarity_matrix[i, j]

        if df.iloc[i]["category"] == df.iloc[j]["category"]:
            same_category_scores.append(score)
        else:
            different_category_scores.append(score)

same_category_scores = np.array(same_category_scores)
different_category_scores = np.array(different_category_scores)

print("Même catégorie   → moyenne:", same_category_scores.mean(), "| min:", same_category_scores.min())
print("Catégorie diff.  → moyenne:", different_category_scores.mean(), "| max:", different_category_scores.max())

Même catégorie   → moyenne: 0.52019554 | min: 0.22775835
Catégorie diff.  → moyenne: 0.28309628 | max: 0.7657879


In [40]:
from sklearn.metrics import roc_curve

# 1 = même catégorie (positif), 0 = catégorie différente (négatif)
labels = np.concatenate([
    np.ones(len(same_category_scores)),
    np.zeros(len(different_category_scores))
])

scores = np.concatenate([
    same_category_scores,
    different_category_scores
])

fpr, tpr, thresholds = roc_curve(labels, scores)

# Indice de Youden : maximise (sensibilité + spécificité - 1)
youden_index = tpr - fpr
best_threshold = thresholds[np.argmax(youden_index)]

print("Seuil optimal trouvé :", round(best_threshold, 3))

Seuil optimal trouvé : 0.442


In [41]:
OPTIMAL_THRESHOLD = round(best_threshold, 3)

# utilisé ensuite dans :
smartdoc_categorize(file_path, threshold=OPTIMAL_THRESHOLD)


Document : voyage_avion.txt
Document le plus proche : voyage_avion.txt
Meilleure catégorie : voyages
Score : 1.0


AttributeError: 'numpy.ndarray' object has no attribute 'append'

In [42]:
# Trouver la paire de documents de catégories différentes 
# avec le score le plus élevé

max_score = -1
max_pair = None

for i in range(n):
    for j in range(i + 1, n):
        if df.iloc[i]["category"] != df.iloc[j]["category"]:
            score = similarity_matrix[i, j]
            if score > max_score:
                max_score = score
                max_pair = (i, j)

i, j = max_pair
print("Score :", round(max_score, 3))
print("Document 1 :", df.iloc[i]["filename"], "→", df.iloc[i]["category"])
print("Document 2 :", df.iloc[j]["filename"], "→", df.iloc[j]["category"])

Score : 0.766
Document 1 : hotel.txt → hebergement_voyage
Document 2 : tourisme_responsable.txt → planification_voyage


In [43]:
from sklearn.metrics import roc_curve
import numpy as np

# 1 = même catégorie (positif), 0 = catégorie différente (négatif)
labels = np.concatenate([
    np.ones(len(same_category_scores)),
    np.zeros(len(different_category_scores))
])

scores = np.concatenate([
    same_category_scores,
    different_category_scores
])

fpr, tpr, thresholds = roc_curve(labels, scores)

# Indice de Youden : maximise (sensibilité + spécificité - 1)
youden_index = tpr - fpr
best_threshold = thresholds[np.argmax(youden_index)]

print("Seuil optimal trouvé :", round(best_threshold, 3))

Seuil optimal trouvé : 0.442


In [44]:
same_correct = (same_category_scores >= best_threshold).sum()
same_total = len(same_category_scores)

diff_correct = (different_category_scores < best_threshold).sum()
diff_total = len(different_category_scores)

print(f"Paires même catégorie bien détectées : {same_correct}/{same_total} ({100*same_correct/same_total:.1f}%)")
print(f"Paires catégorie différente bien détectées : {diff_correct}/{diff_total} ({100*diff_correct/diff_total:.1f}%)")

Paires même catégorie bien détectées : 21/26 (80.8%)
Paires catégorie différente bien détectées : 696/794 (87.7%)


In [45]:
# Quelles paires "même catégorie" sont ratées par le seuil ?
same_category_pairs_scores = []

for i in range(n):
    for j in range(i + 1, n):
        if df.iloc[i]["category"] == df.iloc[j]["category"]:
            score = similarity_matrix[i, j]
            if score < best_threshold:
                same_category_pairs_scores.append({
                    "doc1": df.iloc[i]["filename"],
                    "doc2": df.iloc[j]["filename"],
                    "category": df.iloc[i]["category"],
                    "score": score
                })

pd.DataFrame(same_category_pairs_scores).sort_values("score")

,doc1,doc2,category,score
2,forets.txt,ocean_plastique.txt,biodiversite_ecosystemes,0.227758
3,culture_locale.txt,paris.txt,destinations,0.314164
0,agriculture_durable.txt,recyclage.txt,agriculture_urbanisme,0.317009
1,biodiversite.txt,ocean_plastique.txt,biodiversite_ecosystemes,0.385516
4,voyage_avec_enfants.txt,voyage_avion.txt,voyages,0.390660
